# CWT Macro Economics Sentiment Agent — Interactive Exploration

This notebook walks through each stage of the quantitative trading pipeline:
1. **Data Ingestion**: Querying macroeconomic events, news, and multi-asset price histories
2. **Feature Engineering**: Surprise calculation, calendar features, and rolling regime metrics
3. **Walk-Forward ML Validation**: LightGBM training with out-of-sample forward evaluation
4. **Performance Evaluation**: Sharpe Ratio, Sortino Ratio, Max Drawdown, and Win Rates
5. **Macroeconomic Outlook & Visualizations**


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from config import DATABASE_URL, LOOKBACK_DAYS, ASSET_CLASSES
from src.database.db_manager import DatabaseManager
from src.features.feature_engineering import FeatureEngineer
from src.model.walk_forward import WalkForwardModel
from src.visualization.charts import MacroSentimentCharts
from src.summary.macro_outlook import MacroOutlookGenerator

db = DatabaseManager()
print('Database connection established successfully.')

## 1. Inspect Scraped Macro Events & Sentiment News

In [ ]:
events_df = db.get_macro_events()
print(f'Total Macro Events Ingested: {len(events_df)}')
events_df.head(10)

## 2. Feature Engineering & Matrix Construction

In [ ]:
joined_df = db.get_joined_dataset()
news_df = db.get_news_articles()
fe = FeatureEngineer(joined_df)
feature_matrix = fe.build_feature_matrix(news_df=news_df)
print(f'Feature Matrix Shape: {feature_matrix.shape}')
print(f'Engineered Features: {len(fe.get_feature_importance_names())}')
feature_matrix.head()

## 3. Walk-Forward Modeling & Strategy Backtesting

In [ ]:
model = WalkForwardModel(model_type='lightgbm', train_days=30, test_days=10)
results = model.run(feature_matrix)
metrics = results['overall_metrics']
print('\n--- Overall Model Performance ---')
for k, v in metrics.items():
    print(f'{k:20s}: {v}')

## 4. Multi-Asset Strategy Matrix

In [ ]:
strat_matrix = results['strategy_matrix']
if hasattr(strat_matrix, 'to_dict'):
    strat_df = strat_matrix
else:
    strat_df = pd.DataFrame.from_dict(strat_matrix, orient='index')
strat_df[['sharpe_ratio', 'sortino_ratio', 'max_drawdown', 'win_rate', 'profit_factor', 'cumulative_return']]

## 5. Visualizing Top Predictive Features

In [ ]:
feat_imp = results['feature_importances']
top_feats = pd.Series(feat_imp).sort_values(ascending=False).head(15)
plt.figure(figsize=(10, 6))
top_feats.plot(kind='barh', color='#3b82f6')
plt.title('Top 15 Feature Importances (LightGBM Walk-Forward)')
plt.xlabel('Importance Score')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()